# Tool Calling in LangGraph

## Review

In [3b.reducers.ipynb](3b.reducers.ipynb), we learned how reducers let nodes **accumulate** state (especially messages) rather than overwriting it.

In **wk3** you already used tool calling in LangChain:
- Defined Python functions as tools (using `@tool` or plain functions)
- Bound them to a chat model with `llm.bind_tools([...])`
- Saw the model return `tool_calls` in the `AIMessage`

## Goals

In this notebook we explore how tool calling works **inside a LangGraph graph** and highlight the design differences:

| Aspect | LangChain (wk3) | LangGraph |
|--------|----------------|----------|
| Tool binding | `llm.bind_tools([...])` | Same — unchanged |
| Tool execution | Manual loop or `create_agent` helper | Explicit **ToolNode** in the graph |
| Control flow | Hidden inside agent executor | Visible as **edges** between nodes |
| Deciding "call tool or stop?" | Built into agent loop | Explicit **conditional edge** (`tools_condition`) |
| State management | Implicit message list | Explicit `MessagesState` with reducer |

The key insight: **LangGraph makes the tool-calling loop an explicit, visible graph** rather than a hidden agent executor loop.

In [ ]:
%run ../langchain_common.py

In [ ]:
from typing_extensions import TypedDict
from typing import Annotated
from IPython.display import Image, display
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, AIMessage, HumanMessage, ToolMessage

## Step 1 — Define Tools

This part is **identical** to what you did in wk3. Tools are just Python functions with docstrings that describe their purpose and parameters.

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

def add(a: int, b: int) -> int:
    """Add a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

tools = [multiply, add]

## Step 2 — Bind Tools to the Model

Also the same as wk3. `bind_tools` tells the LLM about available tools so it can decide whether to call one.

In [ ]:
llm_with_tools = llm_noreason.bind_tools(tools)

# Quick test — does the model generate a tool call?
result = llm_with_tools.invoke([HumanMessage(content="What is 3 times 4?")])
print("Tool calls:", result.tool_calls)
print("Content:", result.content)

## Step 3 — The LangChain Way (Recap)

In wk3, the tool execution loop was handled for you by `create_agent`:

```python
from langchain.agents import create_agent
agent = create_agent(llm, tools)
agent.invoke({"messages": [HumanMessage(content="...")]})
```

This hides:
- How the tool call is detected
- How the tool is executed
- How the result is fed back to the model
- When to stop looping

## Step 4 — The LangGraph Way: Explicit Graph

In LangGraph, we make each of these steps **visible as nodes and edges**.

### The Assistant Node

This node invokes the LLM. Its output is an `AIMessage` that *may* contain `tool_calls`.

In [ ]:
def assistant(state: MessagesState):
    """Call the LLM with the current messages."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

### The Tool Node

LangGraph provides `ToolNode` — a pre-built node that:
1. Reads the last `AIMessage` from state
2. Extracts `tool_calls`
3. Executes the matching Python function
4. Returns a `ToolMessage` with the result

You just pass your list of tools:

In [ ]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

### The Conditional Edge

We need a routing function that decides:
- If the model returned tool calls → route to the tool node
- Otherwise → route to END (the model is done)

Let's write this ourselves to see how it works:

In [ ]:
def route_tools(state: MessagesState):
    """Route to tools if the last message has tool_calls, else END."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

# LangGraph also provides a built-in version:
# from langgraph.prebuilt import tools_condition
# (it does exactly the same thing)

### 4d. Assemble the Graph

Now we connect everything:

In [ ]:
builder = StateGraph(MessagesState)

# Add nodes
builder.add_node("assistant", assistant)
builder.add_node("tools", tool_node)

# Add edges
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", route_tools, ["tools", END])
builder.add_edge("tools", "assistant")  # after tool execution, go back to LLM

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

Notice the **loop**: `assistant → tools → assistant`. This is the ReAct pattern made visible:
1. The LLM reasons and (optionally) calls a tool
2. The tool executes and returns a result
3. The LLM sees the result and decides what to do next

## Step 5 — Run It!

In [ ]:
# Test with a tool-requiring input
result = graph.invoke({"messages": [HumanMessage(content="What is 3 multiplied by 7?")]})
print("=== Messages ===")
for m in result["messages"]:
    m.pretty_print()

In [ ]:
# Test with a non-tool input — the model should respond directly
result = graph.invoke({"messages": [HumanMessage(content="Hello, how are you?")]})
print("=== Messages ===")
for m in result["messages"]:
    m.pretty_print()

## Step 6 — Multi-Step Tool Calling

Because of the loop (`tools → assistant`), the model can call **multiple tools in sequence** to solve complex problems. Each iteration appends messages via the `add_messages` reducer.

In [ ]:
# The model may need two tool calls to solve this
result = graph.invoke(
    {"messages": [HumanMessage(content="First add 3 and 4, then multiply the result by 2.")]}
)
print("=== Messages ===")
for m in result["messages"]:
    m.pretty_print()

## Step 7 — Inspecting the Message Flow

Let's look at what types of messages accumulated during the multi-step call. This shows the reducer in action:

In [ ]:
for i, m in enumerate(result["messages"]):
    print(f"{i+1}. {type(m).__name__:15} | tool_calls={getattr(m, 'tool_calls', None) or '—'}")
    print(f"   content: {m.content[:80] if m.content else '(empty)'}")
    print()

## Using the Built-in `tools_condition`

Our hand-written `route_tools` function is helpful for understanding, but in practice you can use the built-in:

In [ ]:
from langgraph.prebuilt import tools_condition

# Rebuild graph using the built-in condition
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")
graph = builder.compile()

# Verify it still works
result = graph.invoke({"messages": [HumanMessage(content="What is 5 + 9?")]})
for m in result["messages"]:
    m.pretty_print()

## Design Philosophy: Why Explicit Graphs?

You might ask: *"If `create_agent` already handled all this, why bother with the graph?"*

The answer is **control and visibility**:

1. **Debuggability** — You can see exactly which node executed and what messages flowed between them
2. **Customization** — You can add nodes *between* the LLM and tool execution (e.g., validation, logging, human approval)
3. **Composition** — You can combine tool-calling subgraphs with other subgraphs (e.g., RAG, routing, multi-agent)
4. **Persistence** — Each step updates state that can be checkpointed and resumed

In the next notebook (**5.router.ipynb**), we'll extend this pattern with conditional routing to handle different types of user input.

## Key Takeaways

| Component | Purpose |
|-----------|--------|
| `llm.bind_tools([...])` | Tell the model what tools exist (same as wk3) |
| Assistant node | Invoke the LLM, get back `AIMessage` with possible `tool_calls` |
| `ToolNode(tools)` | Execute tool calls and return `ToolMessage` results |
| `route_tools` / `tools_condition` | Conditional edge: route to tools or END |
| `tools → assistant` edge | The loop that enables multi-step reasoning |
| `MessagesState` + `add_messages` | Accumulate the full conversation history across iterations |